# ML Lab Exam — All 11 Questions (Adrika Banerjee) — updated list

**How to use on Kaggle:** add the dataset → copy its path from the right sidebar → paste it into `pd.read_csv(...)` (replace `/kaggle/input/YOUR_FOLDER/`).
Every question is **self-contained** — run only the cells of the question you get.

## The one skeleton behind every question
| Step | What you write | Same every time? |
|---|---|---|
| 1 | imports | mostly |
| 2 | `df = pd.read_csv(...)`, `df.head()`, `df.columns` | yes |
| 3 | clean / encode → `X = ...`, `y = ...` | **changes** |
| 4 | `train_test_split(X, y, test_size=0.2, random_state=42)` | yes |
| 5 | `model = ...` → `fit` → `predict` | **changes** |
| 6 | metrics (regression: MAE/MSE/RMSE/R² · classification: confusion matrix + report) | yes |
| 7 | plot (regression: Actual vs Predicted · classification: ROC / tree) | yes |

**If you get a `KeyError`** → run `print(df.columns)` and copy the exact column name.

---
## Q1(a) — Linear Regression: CO2 Emission (`fuel_consumption_dataset.csv`)

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/fuel_consumption_dataset.csv")
print(df.shape)
print(df.columns)
df.head()

In [ ]:
# [3] Features and target
X = df[["ENGINESIZE", "CYLINDERS", "FUELCONSUMPTION_COMB"]]
y = df["CO2EMISSIONS"]

# [4] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# [5] Train and predict
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# [6] Performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

In [ ]:
# [7] Actual vs Predicted plot
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")   # perfect-prediction line
plt.xlabel("Actual CO2 Emissions")
plt.ylabel("Predicted CO2 Emissions")
plt.title("Actual vs Predicted CO2 Emissions")
plt.grid(True)
plt.show()

---
## Q1(b) — Linear Regression: Used Car Selling Price (`used_cars_dataset.csv`)
Text columns (fuel, transmission…) can't go into LinearRegression directly → **OneHotEncoder inside a ColumnTransformer** (same as your Assignment 2).

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/used_cars_dataset.csv")
print(df.shape)
print(df.columns)
df.head()

In [ ]:
# [3] Clean, features and target
df = df.dropna()
target = "selling_price"            # <-- change if your price column has another name (e.g. "price")
y = df[target]
X = df.drop(columns=[target, "name"], errors="ignore")   # car 'name' has too many unique values

num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns
print("Numeric columns    :", list(num_cols))
print("Categorical columns:", list(cat_cols))

# [4] Preprocessor: one-hot encode text columns, pass numbers through
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

# [5] Pipeline = preprocessing + model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# [6] Split, train, predict
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# [7] Performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

In [ ]:
# [8] Actual vs Predicted plot
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Actual vs Predicted Selling Price")
plt.grid(True)
plt.show()

---
## Q2 — Lasso (L1) & Ridge (L2) with hyper-parameter tuning: House Price (`housing_price_dataset.csv`)
⚠️ This dataset is **not** the USA_Housing one from class: it has `SquareFeet, Bedrooms, Bathrooms, Neighborhood, YearBuilt, Price`.
`Neighborhood` is text → needs OneHotEncoder. The code below works for both datasets (it drops `Address` only if it exists).

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/housing_price_dataset.csv")
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())
price_cols = [c for c in df.columns if "price" in c.lower()]
target = price_cols[0] if price_cols else df.columns[-1]   # fallback: last column
print("Target column:", target)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# [3] Features and target
X = df.drop(columns=[target, "Address"], errors="ignore")
y = df[target]

# [4] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

# [5] Preprocessor: scale numbers, one-hot encode text
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# [6] Alpha values to try (hyper-parameter tuning)
alphas = np.logspace(-4, 4, 30)

In [ ]:
# [7] LASSO (L1) + GridSearchCV
lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=100000, random_state=42))
])
lasso_grid = GridSearchCV(lasso_pipeline, param_grid={"model__alpha": alphas},
                          cv=5, scoring="r2", n_jobs=-1)
lasso_grid.fit(X_train, y_train)

lasso_best_alpha = lasso_grid.best_params_["model__alpha"]
lasso_pred = lasso_grid.predict(X_test)
lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_mse = mean_squared_error(y_test, lasso_pred)
lasso_rmse = np.sqrt(lasso_mse)
lasso_r2 = r2_score(y_test, lasso_pred)

print("=== LASSO REGRESSION (L1) ===")
print("Best Alpha:", lasso_best_alpha)
print("MAE :", lasso_mae)
print("MSE :", lasso_mse)
print("RMSE:", lasso_rmse)
print("R2  :", lasso_r2)
print("CV R2:", lasso_grid.best_score_)

In [ ]:
# [8] RIDGE (L2) + GridSearchCV
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])
ridge_grid = GridSearchCV(ridge_pipeline, param_grid={"model__alpha": alphas},
                          cv=5, scoring="r2", n_jobs=-1)
ridge_grid.fit(X_train, y_train)

ridge_best_alpha = ridge_grid.best_params_["model__alpha"]
ridge_pred = ridge_grid.predict(X_test)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_mse = mean_squared_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(ridge_mse)
ridge_r2 = r2_score(y_test, ridge_pred)

print("=== RIDGE REGRESSION (L2) ===")
print("Best Alpha:", ridge_best_alpha)
print("MAE :", ridge_mae)
print("MSE :", ridge_mse)
print("RMSE:", ridge_rmse)
print("R2  :", ridge_r2)
print("CV R2:", ridge_grid.best_score_)

In [ ]:
# [9] Model comparison table
results = pd.DataFrame({
    "Model": ["Lasso (L1)", "Ridge (L2)"],
    "Best Alpha": [lasso_best_alpha, ridge_best_alpha],
    "MAE": [lasso_mae, ridge_mae],
    "MSE": [lasso_mse, ridge_mse],
    "RMSE": [lasso_rmse, ridge_rmse],
    "R2": [lasso_r2, ridge_r2]
})
print("=== MODEL COMPARISON ===")
print(results)

# [10] Actual vs Predicted plots (one per model)
for name, pred in [("Lasso (L1)", lasso_pred), ("Ridge (L2)", ridge_pred)]:
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, pred, alpha=0.3)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", linewidth=2)
    plt.xlabel("Actual House Price")
    plt.ylabel("Predicted House Price")
    plt.title("Actual vs Predicted - " + name)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Q3 — Linear Regression: Selling Price of House (`house_price_dataset.csv`)
⚠️ This is a **different file** from Q2's `housing_price_dataset.csv`. The code below works whatever its columns are:
- finds the price column automatically (`price`, `Price`, `SalePrice`, …)
- drops ID-like / free-text columns (Id, Address, street, date…: almost every value unique)
- fills missing values (median for numbers, most frequent for text)
- one-hot encodes the remaining text columns

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/house_price_dataset.csv")
df.columns = df.columns.str.strip()
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
# [3] Target = the price column
price_cols = [c for c in df.columns if "price" in c.lower()]
target = price_cols[0] if price_cols else df.columns[-1]    # fallback: last column
print("Target column:", target)
df = df.dropna(subset=[target])
y = df[target]
X = df.drop(columns=[target])

# [4] Drop ID columns and text columns where almost every value is different
cat_cols = X.select_dtypes(exclude="number").columns
drop_cols = [c for c in cat_cols if X[c].nunique() > 50]
drop_cols += [c for c in X.columns if c.lower() in ("id", "unnamed: 0")]
X = X.drop(columns=drop_cols)
X = X.dropna(axis=1, how="all")          # remove columns that are completely empty
print("Dropped columns:", drop_cols)

num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns
print("Numeric    :", list(num_cols))
print("Categorical:", list(cat_cols))

# [5] Preprocessor: fill missing values, one-hot encode text
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# [6] Pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# [7] Split, train, predict
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# [8] Performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

In [ ]:
# [9] Actual vs Predicted plot
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Actual vs Predicted House Selling Price")
plt.grid(True)
plt.show()

---
## Q4 — Logistic Regression: Benign vs Malignant (`samples_cancer.csv`)
Class column: **2 = benign → 0**, **4 = malignant → 1**. Some cells contain `?` → convert to number and fill with median.

In [ ]:
# [1] Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, roc_curve, roc_auc_score)

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/samples_cancer.csv")
print(df.columns)
df.head()

In [ ]:
# [3] Clean target
df["class"] = pd.to_numeric(df["class"], errors="coerce")
df["class"] = df["class"].map({2: 0, 4: 1})
df = df.dropna(subset=["class"])

# [4] Features and target
X = df.drop(columns=["class", "ID"], errors="ignore")
y = df["class"].astype(int)
X = X.apply(pd.to_numeric, errors="coerce")    # '?' -> NaN
X = X.fillna(X.median())

# [5] Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# [6] Train and predict
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# [7] Confusion matrix + report
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("-------------------------------")
print("Performance Evaluation:")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

# [8] Individual metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))

In [ ]:
# [9] ROC curve and AUC
y_prob = classifier.predict_proba(X_test)[:, 1]        # probability of class 1 (malignant)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
print("AUC:", auc)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label="Logistic Regression (AUC = %.3f)" % auc)
plt.plot([0, 1], [0, 1], "--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

# [10] Confusion matrix as a picture
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Benign", "Malignant"])
plt.yticks([0, 1], ["Benign", "Malignant"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar()
plt.show()

---
## Q5 — Linear Regression: Mall Customers (`Mall_Customers.csv`)
Predict **Spending Score** from Gender, Age and Annual Income.
Note: R² will be **low (around 0–0.2)**. That is the dataset (spending score barely depends linearly on these), not a bug in your code — say this in the viva.

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/Mall_Customers.csv")
df = df.rename(columns={"Genre": "Gender"})          # some versions call it 'Genre'
print(df.columns)
df.head()

In [ ]:
# [3] Encode Gender, features and target
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})
X = df[["Gender", "Age", "Annual Income (k$)"]]
y = df["Spending Score (1-100)"]

# [4] Split, train, predict
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# [5] Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

# [6] Actual vs Predicted plot
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")
plt.xlabel("Actual Spending Score")
plt.ylabel("Predicted Spending Score")
plt.title("Actual vs Predicted Spending Score")
plt.grid(True)
plt.show()

---
## Q6 — Decision Tree: Mall Customers (`Mall_Customers.csv`) — predict Gender

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree

# [2] Load dataset
dataset = pd.read_csv("/kaggle/input/YOUR_FOLDER/Mall_Customers.csv")
dataset = dataset.rename(columns={"Genre": "Gender"})   # some versions call it 'Genre'
X = dataset.drop(columns=["CustomerID", "Gender"])        # Age, Annual Income, Spending Score
y = dataset["Gender"]                                     # predict Male / Female

# [3] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# [4] Train and predict
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# [5] Evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("-------------------------------")
print("Performance Evaluation:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, y_pred))

# [6] Plot the tree
plt.figure(figsize=(25, 20))
plot_tree(classifier, feature_names=X.columns, class_names=classifier.classes_, filled=True)
plt.show()

---
## Q7 — Decision Tree: Which drug to give a patient (`drug200.csv`)
Columns: `Age, Sex, BP, Cholesterol, Na_to_K, Drug`. Text columns → LabelEncoder.

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/drug200.csv")
print(df.head())
print(df.info())

In [ ]:
# [3] Encode text columns
le_sex = LabelEncoder()
le_bp = LabelEncoder()
le_chol = LabelEncoder()
df["Sex"] = le_sex.fit_transform(df["Sex"])
df["BP"] = le_bp.fit_transform(df["BP"])
df["Cholesterol"] = le_chol.fit_transform(df["Cholesterol"])

# [4] Features and target
X = df.drop("Drug", axis=1)
y = df["Drug"]

# [5] Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# [6] Model
model = DecisionTreeClassifier(criterion="entropy", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# [7] Results
print("Predicted Drugs:", list(y_pred))
print("Actual Drugs   :", list(y_test.values))
print("\nAccuracy of Decision Tree:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# [8] Plot the tree
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=X.columns, class_names=model.classes_, filled=True)
plt.show()

---
## Q8 — SVM: Mall Customers (`Mall_Customers.csv`) — predict Gender, all kernels
Scaling added (wasn't in class): SVM measures distances, so Income (15–137) would otherwise dominate.

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# [2] Load dataset
dataset = pd.read_csv("/kaggle/input/YOUR_FOLDER/Mall_Customers.csv")
dataset = dataset.rename(columns={"Genre": "Gender"})   # some versions call it 'Genre'
X = dataset.drop(columns=["CustomerID", "Gender"])        # Age, Annual Income, Spending Score
y = dataset["Gender"]                                     # predict Male / Female

# [3] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# [4] Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# [5] Try every kernel
kernels = [
    ("Linear",            SVC(kernel="linear")),
    ("Polynomial deg=2",  SVC(kernel="poly", degree=2)),
    ("Polynomial deg=3",  SVC(kernel="poly", degree=3)),
    ("RBF",               SVC(kernel="rbf")),
    ("Sigmoid",           SVC(kernel="sigmoid")),
]
accuracies = []
for name, classifier in kernels:
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    print("==========", name, "kernel ==========")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("-------------------------------")
    print("Performance Evaluation:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    
    accuracies.append(accuracy_score(y_test, y_pred))

# [6] Summary
print(pd.DataFrame({"Kernel": [k[0] for k in kernels], "Accuracy": accuracies}))

---
## Q9 — Naive Bayes 1: **Multinomial NB** (`Mall_Customers.csv`)
MultinomialNB needs **non-negative** features → do **not** scale.

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.naive_bayes import MultinomialNB

# [2] Load dataset
dataset = pd.read_csv("/kaggle/input/YOUR_FOLDER/Mall_Customers.csv")
dataset = dataset.rename(columns={"Genre": "Gender"})   # some versions call it 'Genre'
X = dataset.drop(columns=["CustomerID", "Gender"])        # Age, Annual Income, Spending Score
y = dataset["Gender"]                                     # predict Male / Female

# [3] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# [4] Train and predict
classifier = MultinomialNB()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# [5] Evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("-------------------------------")
print("Performance Evaluation:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, y_pred))

---
## Q10 — Naive Bayes 2: **Gaussian NB** (`Mall_Customers.csv`)
GaussianNB = for continuous numbers (assumes each feature is bell-curve shaped per class).

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.naive_bayes import GaussianNB

# [2] Load dataset
dataset = pd.read_csv("/kaggle/input/YOUR_FOLDER/Mall_Customers.csv")
dataset = dataset.rename(columns={"Genre": "Gender"})   # some versions call it 'Genre'
X = dataset.drop(columns=["CustomerID", "Gender"])        # Age, Annual Income, Spending Score
y = dataset["Gender"]                                     # predict Male / Female

# [3] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# [4] Train and predict
classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# [5] Evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("-------------------------------")
print("Performance Evaluation:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, y_pred))

---
## Q11 — Naive Bayes 3: **Bernoulli NB** (`Mall_Customers.csv`)
BernoulliNB works on **0/1 features**. By default it turns every value > 0 into 1 — all Mall features are > 0, so
every row becomes `[1,1,1]` and it just predicts the majority class (that's why your class output had Male recall = 0.00).
Fix: convert each feature to 0/1 using its median (part b).

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.naive_bayes import BernoulliNB

# [2] Load dataset
dataset = pd.read_csv("/kaggle/input/YOUR_FOLDER/Mall_Customers.csv")
dataset = dataset.rename(columns={"Genre": "Gender"})   # some versions call it 'Genre'
X = dataset.drop(columns=["CustomerID", "Gender"])        # Age, Annual Income, Spending Score
y = dataset["Gender"]                                     # predict Male / Female

# [3] Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# [4a] Default BernoulliNB (same as class)
classifier = BernoulliNB()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print("##### (a) Default BernoulliNB #####")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("-------------------------------")
print("Performance Evaluation:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, y_pred))

# [4b] Better: binarize each feature at its training median (above median = 1, else 0)
median = X_train.median()
X_train_bin = (X_train > median).astype(int)
X_test_bin = (X_test > median).astype(int)

classifier = BernoulliNB(binarize=None)     # data is already 0/1
classifier.fit(X_train_bin, y_train)
y_pred = classifier.predict(X_test_bin)
print("\n##### (b) BernoulliNB on median-binarized features #####")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("-------------------------------")
print("Performance Evaluation:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, y_pred))

---
## Extra (old Q3, removed from the new list) — Linear Regression with Gradient Descent from scratch: Salary (`salary_dataset.csv`)
No sklearn model, no sklearn metrics — only numpy.
**Idea:** line `y = m·x + c`. Start with m = c = 0. Every epoch: find which way the error goes down (gradient) and take a small step (learning rate) that way.

| Formula | Code |
|---|---|
| Loss (MSE) = mean((y − ŷ)²) | `np.mean((y - y_pred)**2)` |
| ∂L/∂m = −2/n · Σ x(y − ŷ) | `dm = (-2/n) * np.sum(x * (y - y_pred))` |
| ∂L/∂c = −2/n · Σ (y − ŷ) | `dc = (-2/n) * np.sum(y - y_pred)` |
| update | `m = m - lr*dm`, `c = c - lr*dc` |

In [ ]:
# [1] Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# [2] Load dataset
df = pd.read_csv("/kaggle/input/YOUR_FOLDER/salary_dataset.csv")
print(df.columns)
df.head()

In [ ]:
# [3] Feature and target
x = df["YearsExperience"].values
y = df["Salary"].values

# [4] Train-test split (manual, 80/20)
np.random.seed(42)
idx = np.random.permutation(len(x))
split = int(0.8 * len(x))
x_train, x_test = x[idx[:split]], x[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]
print("Train:", len(x_train), " Test:", len(x_test))

# [5] Metric functions (written by hand)
def mae(y, y_pred):
    return np.mean(np.abs(y - y_pred))

def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)

def r2(y, y_pred):
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - ss_res / ss_tot

# [6] Gradient descent
def gradient_descent(x, y, lr, epochs):
    m, c = 0.0, 0.0
    n = len(x)
    history = []                                   # (m, c, loss) after every epoch
    for epoch in range(epochs):
        y_pred = m * x + c
        dm = (-2 / n) * np.sum(x * (y - y_pred))   # slope of loss w.r.t. m
        dc = (-2 / n) * np.sum(y - y_pred)         # slope of loss w.r.t. c
        m = m - lr * dm
        c = c - lr * dc
        history.append((m, c, mse(y, m * x + c)))
    return m, c, history

In [ ]:
# [7] Hyper-parameter tuning: try every (learning rate, epochs) pair, keep the best R2
learning_rates = [0.0001, 0.001, 0.005, 0.01, 0.02]
epoch_list = [100, 500, 1000, 2000]

best_lr, best_epochs, best_r2 = None, None, -np.inf
print("lr        epochs   test R2")
for lr in learning_rates:
    for epochs in epoch_list:
        m, c, _ = gradient_descent(x_train, y_train, lr, epochs)
        score = r2(y_test, m * x_test + c)
        print(f"{lr:<9} {epochs:<8} {score:.4f}")
        if score > best_r2:
            best_lr, best_epochs, best_r2 = lr, epochs, score

print("\nBest learning rate:", best_lr)
print("Best epochs       :", best_epochs)

In [ ]:
# [8] Final model with best hyper-parameters
m, c, history = gradient_descent(x_train, y_train, best_lr, best_epochs)
y_pred = m * x_test + c

print("Final equation: Salary = %.2f * YearsExperience + %.2f" % (m, c))
print("MAE :", mae(y_test, y_pred))
print("MSE :", mse(y_test, y_pred))
print("RMSE:", np.sqrt(mse(y_test, y_pred)))
print("R2  :", r2(y_test, y_pred))

In [ ]:
# [9a] Hypothesis line after EVERY epoch (light lines) + final line (red)
x_line = np.linspace(x.min(), x.max(), 100)
plt.figure(figsize=(8, 6))
plt.scatter(x_train, y_train, color="blue", label="Training data", zorder=3)
for (m_e, c_e, _) in history:
    plt.plot(x_line, m_e * x_line + c_e, color="gray", alpha=0.05)
plt.plot(x_line, m * x_line + c, color="red", linewidth=2, label="Final hypothesis")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Hypothesis after each epoch")
plt.legend()
plt.grid(True)
plt.show()

# [9b] Snapshots at a few epochs (easier to see the line moving)
snap = np.unique(np.geomspace(1, best_epochs, 8).astype(int)) - 1
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, e in zip(axes.flat, snap):
    m_e, c_e, loss_e = history[e]
    ax.scatter(x_train, y_train, s=15)
    ax.plot(x_line, m_e * x_line + c_e, "r")
    ax.set_title(f"Epoch {e + 1}  MSE={loss_e:.2e}")
plt.tight_layout()
plt.show()

# [9c] Loss vs epoch
plt.figure(figsize=(7, 4))
plt.plot([h[2] for h in history])
plt.xlabel("Epoch")
plt.ylabel("MSE (training)")
plt.title("Loss decreasing over epochs")
plt.grid(True)
plt.show()

# [9d] Actual vs Predicted
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual Salary")
plt.ylabel("Predicted Salary")
plt.title("Actual vs Predicted Salary")
plt.grid(True)
plt.show()